# Search + URN Navigation

This notebook demonstrates Scaife JSON search and CTS navigation from ordered valid references. The live Perseus CTS service may return malformed HTML for `GetPrevNextUrn`, so the navigation example derives neighbors from `GetValidReff`, matching the fallback strategy implemented by the MCP server.

The edition URN is an example recorded from the live CTS inventory. Discover the current edition before reusing it in other workflows.

We use `%pip` instead of `!pip` because `%pip` installs into the current Jupyter kernel.

In [1]:
%pip install --quiet httpx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import xml.etree.ElementTree as ET

import httpx

CTS_BASE = "https://www.perseus.tufts.edu/hopper/CTS"
SEARCH_BASE = "https://scaife.perseus.org/search/json/"

In [3]:
query = "μῆνιν"
search = httpx.get(
    SEARCH_BASE,
    params={"q": query, "kind": "form", "type": "library", "page_num": 1},
    timeout=20.0,
)
search.raise_for_status()
print(json.dumps(search.json(), ensure_ascii=False, indent=2)[:1200])

{
  "results": [
    {
      "passage": {
        "url": "/reader/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/",
        "json_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/json/",
        "text_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/text/",
        "text": {
          "url": "/library/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/",
          "json_url": "/library/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/json/",
          "text_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/text/",
          "ancestors": [
            {
              "url": "/library/urn:cts:greekLit:tlg2045/",
              "json_url": "/library/urn:cts:greekLit:tlg2045/json/",
              "text_url": "/library/passage/urn:cts:greekLit:tlg2045/text/",
              "urn": "urn:cts:greekLit:tlg2045",
              "label": "Nonnus of Panopolis"
            },
            {
              "url": "/library/urn:cts:greekL

In [4]:
edition = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"
current = f"{edition}:1.10"
refs = httpx.get(
    CTS_BASE,
    params={"request": "GetValidReff", "urn": edition},
    timeout=20.0,
)
refs.raise_for_status()

root = ET.fromstring(refs.text)
urns = [element.text for element in root.iter() if element.tag.rsplit("}", 1)[-1] == "urn"]
index = urns.index(current)
print("previous:", urns[index - 1])
print("current: ", current)
print("next:    ", urns[index + 1])

previous: urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.9
current:  urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10
next:     urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.11


# Notebook version

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 3, 2026</td>
    </tr>
  </table>
</div>